# NeuroScope AI - Notebook 13: RAG Pipeline

Retrieval-Augmented Generation for evidence-based clinical intelligence.
Powers Agent 5 with real medical literature instead of static rules.

**What this notebook builds:**
1. PubMed retrieval -- fetch abstracts via NCBI E-utilities API
2. NCCN/ESMO guideline structuring -- JSON knowledge base
3. Text chunking and embedding -- sentence-transformers
4. Vector store -- FAISS for fast similarity search
5. RAG query pipeline -- retrieve + Claude API synthesis
6. Evidence-based Agent 5 upgrade

**How it improves Agent 5:**
- Before NB13: static NCCN rule lookup (hardcoded)
- After NB13: retrieves relevant papers + guidelines, passes to Claude as context
- Claude reasons over real evidence, not just lookup tables
- Cites specific papers and guideline versions

---

## Cell 1 - Imports & Config

In [1]:
import os, sys, json, time, warnings, hashlib
import numpy as np
import requests
warnings.filterwarnings('ignore')

# FAISS for vector search
try:
    import faiss
    FAISS_AVAILABLE = True
    print(f'FAISS     : OK')
except ImportError:
    FAISS_AVAILABLE = False
    print('FAISS not installed -- run: pip install faiss-cpu')

# sentence-transformers for embeddings
try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
    print(f'SentenceTransformers: OK')
except ImportError:
    ST_AVAILABLE = False
    print('sentence-transformers not installed -- run: pip install sentence-transformers')

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
RAG  = os.path.join(BASE, 'rag')
OUT  = os.path.join(BASE, 'outputs', 'nb13_rag')
os.makedirs(os.path.join(RAG, 'corpus'),    exist_ok=True)
os.makedirs(os.path.join(RAG, 'index'),     exist_ok=True)
os.makedirs(os.path.join(RAG, 'cache'),     exist_ok=True)
os.makedirs(os.path.join(RAG, 'guidelines'),exist_ok=True)
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, BASE)

EMBED_MODEL = 'all-MiniLM-L6-v2'   # 80MB, fast, good for medical text
EMBED_DIM   = 384

print(f'RAG dir   : {RAG}')
print('Imports OK')

FAISS     : OK
SentenceTransformers: OK
RAG dir   : C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\rag
Imports OK


---
## Cell 2 - PubMed Retrieval

In [2]:
import os, json, time, hashlib, requests

BASE  = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
CACHE = os.path.join(BASE, 'rag', 'cache')

PUBMED_BASE   = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
PUBMED_SEARCH = f'{PUBMED_BASE}/esearch.fcgi'
PUBMED_FETCH  = f'{PUBMED_BASE}/efetch.fcgi'


def pubmed_search(query, max_results=10, retries=3):
    """
    Search PubMed for abstracts matching query.
    Returns list of article dicts with title, abstract, authors, year, pmid.
    Results are cached to disk to avoid redundant API calls.
    """
    cache_key  = hashlib.md5(f'{query}_{max_results}'.encode()).hexdigest()[:12]
    cache_path = os.path.join(CACHE, f'pubmed_{cache_key}.json')

    if os.path.exists(cache_path):
        with open(cache_path, encoding='utf-8') as f:
            cached = json.load(f)
        print(f'  [cache] {len(cached)} results for: {query[:60]}')
        return cached

    articles = []
    for attempt in range(retries):
        try:
            # Step 1: Search for PMIDs
            search_params = {
                'db'      : 'pubmed',
                'term'    : query,
                'retmax'  : max_results,
                'retmode' : 'json',
                'sort'    : 'relevance',
            }
            r = requests.get(PUBMED_SEARCH, params=search_params, timeout=15)
            r.raise_for_status()
            pmids = r.json()['esearchresult']['idlist']

            if not pmids:
                print(f'  No results for: {query}')
                return []

            # Step 2: Fetch abstracts
            fetch_params = {
                'db'      : 'pubmed',
                'id'      : ','.join(pmids),
                'retmode' : 'xml',
                'rettype' : 'abstract',
            }
            fr = requests.get(PUBMED_FETCH, params=fetch_params, timeout=30)
            fr.raise_for_status()

            # Parse XML
            import xml.etree.ElementTree as ET
            root = ET.fromstring(fr.text)

            for article in root.findall('.//PubmedArticle'):
                try:
                    title  = article.findtext('.//ArticleTitle', '')
                    abstract_parts = article.findall('.//AbstractText')
                    abstract = ' '.join(p.text or '' for p in abstract_parts if p.text)
                    pmid   = article.findtext('.//PMID', '')
                    year_el= article.find('.//PubDate/Year')
                    year   = year_el.text if year_el is not None else 'unknown'
                    authors= []
                    for auth in article.findall('.//Author')[:3]:
                        ln = auth.findtext('LastName', '')
                        fn = auth.findtext('ForeName', '')
                        if ln:
                            authors.append(f'{ln} {fn}'.strip())

                    if abstract:
                        articles.append({
                            'pmid'    : pmid,
                            'title'   : title,
                            'abstract': abstract[:2000],
                            'authors' : authors,
                            'year'    : year,
                            'source'  : 'pubmed',
                            'url'     : f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/',
                        })
                except Exception:
                    continue

            break   # success

        except Exception as e:
            print(f'  PubMed attempt {attempt+1} failed: {e}')
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    # Cache results
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)

    print(f'  [pubmed] {len(articles)} results for: {query[:60]}')
    return articles


# ── Fetch papers for all 6 cancer types ──────────────────────────────────────
PUBMED_QUERIES = {
    'brain'  : 'glioblastoma temozolomide radiotherapy treatment outcomes[tiab]',
    'lung'   : 'lung cancer nodule detection deep learning CT 2024[pdat]',
    'breast' : 'breast cancer mammography AI detection 2024[pdat]',
    'liver'  : 'hepatocellular carcinoma LI-RADS treatment 2024[pdat]',
    'skin'   : 'melanoma dermoscopy deep learning classification 2024[pdat]',
    'spine'  : 'lumbar spine degenerative stenosis MRI classification 2024[pdat]',
}

all_articles = []
print('Fetching PubMed abstracts...')
for cancer, query in PUBMED_QUERIES.items():
    articles = pubmed_search(query, max_results=5)
    for a in articles:
        a['cancer_type'] = cancer
    all_articles.extend(articles)
    time.sleep(0.5)   # rate limit: 3 requests/sec without API key

print(f'\nTotal articles fetched: {len(all_articles)}')

corpus_path = os.path.join(BASE, 'rag', 'corpus', 'pubmed_corpus.json')
with open(corpus_path, 'w', encoding='utf-8') as f:
    json.dump(all_articles, f, indent=2, ensure_ascii=False)
print(f'Corpus saved: {corpus_path}')

Fetching PubMed abstracts...
  No results for: glioblastoma temozolomide radiotherapy treatment outcomes[tiab]
  [cache] 5 results for: lung cancer nodule detection deep learning CT 2024[pdat]
  [cache] 5 results for: breast cancer mammography AI detection 2024[pdat]
  [cache] 3 results for: hepatocellular carcinoma LI-RADS treatment 2024[pdat]
  [cache] 4 results for: melanoma dermoscopy deep learning classification 2024[pdat]
  [cache] 5 results for: lumbar spine degenerative stenosis MRI classification 2024[p

Total articles fetched: 22
Corpus saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\rag\corpus\pubmed_corpus.json


---
## Cell 3 - NCCN/ESMO Guideline Knowledge Base

In [3]:
import os, json

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'

# Structured NCCN/ESMO guidelines as JSON knowledge base
# These are public guideline summaries -- not copyrighted clinical text
GUIDELINES = {
    'brain': [
        {
            'id'      : 'nccn_brain_gbm_2024',
            'source'  : 'NCCN Guidelines CNS Tumors v1.2024',
            'tumor'   : 'glioblastoma',
            'grade'   : 'Grade IV',
            'title'   : 'GBM Standard of Care -- Stupp Protocol',
            'content' : (
                'Newly diagnosed GBM: maximal safe surgical resection followed by '
                'concurrent radiotherapy (60Gy/30 fractions) plus temozolomide (75mg/m2/day). '
                'Adjuvant TMZ 150-200mg/m2 day 1-5 every 28 days for 6 cycles. '
                'MGMT promoter methylation testing required -- methylated patients '
                'have significantly better response to TMZ (OS 21.7 vs 12.7 months). '
                'Tumor treating fields (Optune) category 1 recommendation for unmethylated. '
                'Bevacizumab approved for recurrent GBM (PFS benefit, no OS benefit). '
                'IDH1/2 mutation testing required for classification.'
            ),
            'evidence_level': 'Category 1',
            'tags'    : ['glioblastoma', 'GBM', 'temozolomide', 'radiotherapy', 'MGMT', 'IDH'],
        },
        {
            'id'      : 'nccn_brain_lgg_2024',
            'source'  : 'NCCN Guidelines CNS Tumors v1.2024',
            'tumor'   : 'low-grade glioma',
            'grade'   : 'Grade II',
            'title'   : 'Low-Grade Glioma Management',
            'content' : (
                'IDH-mutant grade 2 glioma: observation acceptable for low-risk '
                '(age <40, complete resection, no contrast enhancement). '
                'High-risk: RT 45-54Gy + PCV chemotherapy (category 1). '
                '1p/19q codeletion defines oligodendroglioma -- better prognosis. '
                'Temozolomide alternative to PCV for IDH-mutant astrocytoma. '
                'MRI surveillance every 3-6 months first 5 years.'
            ),
            'evidence_level': 'Category 1',
            'tags'    : ['low-grade glioma', 'IDH', '1p19q', 'PCV', 'oligodendroglioma'],
        },
        {
            'id'      : 'nccn_meningioma_2024',
            'source'  : 'NCCN Guidelines CNS Tumors v1.2024',
            'tumor'   : 'meningioma',
            'grade'   : 'Grade I',
            'title'   : 'Meningioma Management',
            'content' : (
                'Grade I meningioma: observation for asymptomatic, small (<3cm). '
                'Surgery for symptomatic or enlarging tumors (Simpson grade I-II resection target). '
                'Stereotactic radiosurgery (SRS) for surgically inaccessible or residual disease. '
                'Grade II: surgery + adjuvant RT 54-60Gy. '
                'Grade III: surgery + RT, consider clinical trial.'
            ),
            'evidence_level': 'Category 2A',
            'tags'    : ['meningioma', 'SRS', 'radiosurgery', 'Simpson'],
        },
    ],
    'breast': [
        {
            'id'      : 'nccn_breast_invasive_2024',
            'source'  : 'NCCN Guidelines Breast Cancer v4.2024',
            'tumor'   : 'invasive breast carcinoma',
            'title'   : 'Invasive Breast Cancer -- Surgical and Systemic Therapy',
            'content' : (
                'Stage I-II invasive breast cancer: breast conservation (lumpectomy + RT) '
                'equivalent to mastectomy for local control (category 1). '
                'Sentinel lymph node biopsy standard for clinically node-negative. '
                'HER2-positive: trastuzumab + pertuzumab + taxane-based chemo (THP). '
                'ER/PR-positive, HER2-negative: endocrine therapy (tamoxifen or AI) +/- CDK4/6 inhibitor. '
                'Triple-negative: pembrolizumab + chemo for PD-L1 positive. '
                'Neoadjuvant chemo for tumors >2cm or node-positive (pathologic CR improves survival). '
                'Oncotype DX for node-negative, ER+, HER2- to guide chemo decision.'
            ),
            'evidence_level': 'Category 1',
            'tags'    : ['breast cancer', 'HER2', 'trastuzumab', 'tamoxifen', 'CDK4/6', 'TNBC'],
        },
    ],
    'skin': [
        {
            'id'      : 'nccn_melanoma_2024',
            'source'  : 'NCCN Guidelines Melanoma v2.2024',
            'tumor'   : 'melanoma',
            'title'   : 'Cutaneous Melanoma Treatment',
            'content' : (
                'Stage I-II: wide local excision with margins based on Breslow thickness '
                '(0.8-1mm: 1cm margin; >2mm: 2cm margin). '
                'SLNB for tumors >0.8mm or with ulceration/mitosis. '
                'Stage III-IV: pembrolizumab or nivolumab first-line immunotherapy (category 1). '
                'BRAF V600E/K mutation (50% of melanomas): dabrafenib + trametinib targeted therapy. '
                'Ipilimumab + nivolumab for high-risk stage III (category 1). '
                'Adjuvant pembrolizumab after resection reduces recurrence 35%.'
            ),
            'evidence_level': 'Category 1',
            'tags'    : ['melanoma', 'pembrolizumab', 'nivolumab', 'BRAF', 'dabrafenib', 'immunotherapy'],
        },
    ],
    'lung': [
        {
            'id'      : 'nccn_lung_nsclc_2024',
            'source'  : 'NCCN Guidelines NSCLC v3.2024',
            'tumor'   : 'NSCLC',
            'title'   : 'Non-Small Cell Lung Cancer -- Stage-Based Treatment',
            'content' : (
                'Stage I-II resectable NSCLC: lobectomy standard (category 1); '
                'SBRT for medically inoperable. '
                'Stage III unresectable: concurrent chemo-RT (carboplatin/paclitaxel + RT 60Gy). '
                'Durvalumab consolidation after CRT (PACIFIC trial, OS benefit). '
                'Stage IV: molecular testing mandatory (EGFR, ALK, ROS1, KRAS G12C, MET, RET, NTRK, PD-L1). '
                'EGFR-mutant: osimertinib first-line (category 1). '
                'ALK-positive: alectinib first-line. '
                'PD-L1 >=50%: pembrolizumab monotherapy. '
                'Lung-RADS 4B: biopsy recommended.'
            ),
            'evidence_level': 'Category 1',
            'tags'    : ['NSCLC', 'EGFR', 'ALK', 'pembrolizumab', 'osimertinib', 'Lung-RADS'],
        },
    ],
    'liver': [
        {
            'id'      : 'nccn_hcc_2024',
            'source'  : 'NCCN Guidelines Hepatocellular Carcinoma v2.2024',
            'tumor'   : 'hepatocellular carcinoma',
            'title'   : 'HCC Treatment by Stage (BCLC)',
            'content' : (
                'BCLC-0/A (early): curative intent -- resection, transplant, or ablation. '
                'Resection if adequate liver reserve (Child-Pugh A). '
                'Milan criteria met: liver transplant preferred for solitary <5cm or 3 nodules <3cm. '
                'RFA/MWA for unresectable BCLC-A. '
                'BCLC-B (intermediate): TACE (transarterial chemoembolization). '
                'BCLC-C (advanced): atezolizumab + bevacizumab first-line (IMbrave150, category 1). '
                'Sorafenib/lenvatinib alternatives. '
                'LI-RADS 5 lesion = HCC -- no biopsy needed for treatment decision.'
            ),
            'evidence_level': 'Category 1',
            'tags'    : ['HCC', 'TACE', 'sorafenib', 'BCLC', 'LI-RADS', 'ablation'],
        },
    ],
    'spine': [
        {
            'id'      : 'nass_spine_2024',
            'source'  : 'NASS Evidence-Based Guidelines 2024',
            'tumor'   : 'degenerative lumbar stenosis',
            'title'   : 'Lumbar Spinal Stenosis Management',
            'content' : (
                'Neurogenic claudication: conservative management first 6-12 weeks '
                '(physical therapy, NSAIDs, epidural steroid injection). '
                'Surgical decompression (laminectomy) for failed conservative therapy '
                'or progressive neurological deficit. '
                'Cauda equina syndrome: emergency surgical decompression within 24-48 hours. '
                'Spondylolisthesis with instability: fusion in addition to decompression. '
                'SPORT trial: surgery superior to conservative at 2 years for severe stenosis. '
                'Neural foraminal narrowing Severe: consider ESI before surgery.'
            ),
            'evidence_level': 'Grade A',
            'tags'    : ['stenosis', 'laminectomy', 'ESI', 'cauda equina', 'fusion', 'SPORT'],
        },
    ],
}

# Flatten to list of documents
guideline_docs = []
for cancer, docs in GUIDELINES.items():
    for doc in docs:
        doc['cancer_type'] = cancer
        guideline_docs.append(doc)

# Save
gl_path = os.path.join(BASE, 'rag', 'guidelines', 'nccn_esmo_guidelines.json')
with open(gl_path, 'w', encoding='utf-8') as f:
    json.dump(guideline_docs, f, indent=2, ensure_ascii=False)

print(f'Guidelines: {len(guideline_docs)} entries across {len(GUIDELINES)} cancer types')
for cancer, docs in GUIDELINES.items():
    print(f'  {cancer:8s}: {len(docs)} guidelines')
print(f'Saved: {gl_path}')

Guidelines: 8 entries across 6 cancer types
  brain   : 3 guidelines
  breast  : 1 guidelines
  skin    : 1 guidelines
  lung    : 1 guidelines
  liver   : 1 guidelines
  spine   : 1 guidelines
Saved: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\rag\guidelines\nccn_esmo_guidelines.json


---
## Cell 4 - Text Chunking & Embedding

In [4]:
import os, json, numpy as np

BASE   = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
CORPUS = os.path.join(BASE, 'rag', 'corpus', 'pubmed_corpus.json')
GL     = os.path.join(BASE, 'rag', 'guidelines', 'nccn_esmo_guidelines.json')


def chunk_text(text, chunk_size=400, overlap=50):
    """
    Split text into overlapping chunks for embedding.
    Overlap ensures context isn't lost at boundaries.
    """
    words  = text.split()
    chunks = []
    i      = 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks


def build_document_corpus():
    """
    Combine PubMed articles + NCCN guidelines into unified corpus.
    Each document = one embeddable chunk with metadata.
    """
    corpus = []

    # PubMed articles
    if os.path.exists(CORPUS):
        with open(CORPUS, encoding='utf-8') as f:
            articles = json.load(f)
        for art in articles:
            text = f"{art['title']}. {art['abstract']}"
            for chunk in chunk_text(text):
                corpus.append({
                    'text'       : chunk,
                    'source'     : f"PubMed PMID:{art['pmid']}",
                    'title'      : art['title'],
                    'year'       : art['year'],
                    'cancer_type': art.get('cancer_type', 'general'),
                    'type'       : 'pubmed',
                    'url'        : art.get('url', ''),
                })

    # NCCN/ESMO guidelines
    if os.path.exists(GL):
        with open(GL, encoding='utf-8') as f:
            guidelines = json.load(f)
        for gl in guidelines:
            text = f"{gl['title']}. {gl['content']}"
            for chunk in chunk_text(text, chunk_size=300):
                corpus.append({
                    'text'       : chunk,
                    'source'     : gl['source'],
                    'title'      : gl['title'],
                    'cancer_type': gl['cancer_type'],
                    'type'       : 'guideline',
                    'evidence'   : gl.get('evidence_level', ''),
                    'tags'       : gl.get('tags', []),
                })

    return corpus


def embed_corpus(corpus, model):
    """
    Embed all documents using sentence-transformers.
    Returns numpy array [n_docs, embed_dim].
    """
    texts      = [doc['text'] for doc in corpus]
    print(f'Embedding {len(texts)} chunks...')
    embeddings = model.encode(
        texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True   # L2 norm for cosine similarity
    )
    return embeddings.astype(np.float32)


corpus = build_document_corpus()
print(f'Corpus built: {len(corpus)} chunks')
by_type = {}
for doc in corpus:
    t = doc.get('type', 'other')
    by_type[t] = by_type.get(t, 0) + 1
for t, n in by_type.items():
    print(f'  {t:12s}: {n} chunks')

# Embed if sentence-transformers available
embeddings = None
if ST_AVAILABLE and corpus:
    print('\nLoading embedding model...')
    embed_model = SentenceTransformer(EMBED_MODEL)
    embeddings  = embed_corpus(corpus, embed_model)
    print(f'Embeddings shape: {embeddings.shape}')
    # Save
    np.save(os.path.join(BASE, 'rag', 'index', 'embeddings.npy'), embeddings)
    with open(os.path.join(BASE, 'rag', 'index', 'corpus_meta.json'), 'w',
              encoding='utf-8') as f:
        json.dump(corpus, f, ensure_ascii=False)
    print('Embeddings saved')
else:
    print('sentence-transformers not available -- skipping embedding')

Corpus built: 30 chunks
  pubmed      : 22 chunks
  guideline   : 8 chunks

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8591.64it/s]


Embedding 30 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]

Embeddings shape: (30, 384)
Embeddings saved


---
## Cell 5 - FAISS Vector Store

In [5]:
import os, json, numpy as np

BASE      = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
INDEX_DIR = os.path.join(BASE, 'rag', 'index')


class MedicalVectorStore:
    """
    FAISS-backed vector store for medical literature retrieval.
    Falls back to keyword search if FAISS not available.
    """

    def __init__(self, embed_dim=384):
        self.embed_dim = embed_dim
        self.corpus    = []
        self.index     = None
        self.embed_model = None

    def load(self, index_dir=INDEX_DIR):
        """Load pre-built index and corpus from disk."""
        emb_path    = os.path.join(index_dir, 'embeddings.npy')
        corpus_path = os.path.join(index_dir, 'corpus_meta.json')

        if not os.path.exists(emb_path) or not os.path.exists(corpus_path):
            print('No pre-built index found -- run Cell 4 first')
            return False

        embeddings = np.load(emb_path)
        with open(corpus_path, encoding='utf-8') as f:
            self.corpus = json.load(f)

        if FAISS_AVAILABLE:
            self.index = faiss.IndexFlatIP(self.embed_dim)   # inner product = cosine for L2-normed
            self.index.add(embeddings)
            print(f'FAISS index loaded: {self.index.ntotal} vectors')
        else:
            self.embeddings = embeddings
            print(f'Numpy fallback: {len(embeddings)} vectors')

        if ST_AVAILABLE:
            self.embed_model = SentenceTransformer(EMBED_MODEL)

        return True

    def search(self, query, top_k=5, cancer_type=None):
        """
        Retrieve top_k most relevant chunks for a query.
        Optionally filter by cancer_type.
        """
        if not self.corpus:
            return self._keyword_search(query, top_k, cancer_type)

        if self.embed_model is None or (not FAISS_AVAILABLE and
                                        not hasattr(self, 'embeddings')):
            return self._keyword_search(query, top_k, cancer_type)

        # Embed query
        q_emb = self.embed_model.encode(
            [query], normalize_embeddings=True
        ).astype(np.float32)

        if FAISS_AVAILABLE:
            scores, indices = self.index.search(q_emb, top_k * 3)
            results = [
                {**self.corpus[i], 'score': float(scores[0][j])}
                for j, i in enumerate(indices[0]) if i < len(self.corpus)
            ]
        else:
            sims    = (self.embeddings @ q_emb.T).squeeze()
            indices = np.argsort(sims)[::-1][:top_k * 3]
            results = [
                {**self.corpus[i], 'score': float(sims[i])}
                for i in indices
            ]

        # Filter by cancer type
        if cancer_type:
            results = [r for r in results
                       if r.get('cancer_type') == cancer_type
                       or r.get('cancer_type') == 'general']

        return results[:top_k]

    def _keyword_search(self, query, top_k=5, cancer_type=None):
        """Keyword fallback when embeddings unavailable."""
        query_words = set(query.lower().split())
        scored = []
        for i, doc in enumerate(self.corpus):
            if cancer_type and doc.get('cancer_type') not in (cancer_type, 'general', None):
                continue
            text_words = set(doc['text'].lower().split())
            score = len(query_words & text_words) / max(len(query_words), 1)
            if score > 0:
                scored.append({**doc, 'score': score})
        scored.sort(key=lambda x: x['score'], reverse=True)
        return scored[:top_k]


# Initialize and load vector store
vector_store = MedicalVectorStore(EMBED_DIM)
loaded = vector_store.load()

if loaded:
    # Test query
    test_query = 'glioblastoma treatment temozolomide'
    results = vector_store.search(test_query, top_k=3, cancer_type='brain')
    print(f'\nTest query: "{test_query}"')
    print(f'Results: {len(results)}')
    for r in results:
        print(f'  [{r["score"]:.3f}] {r["source"][:50]} -- {r["text"][:80]}...')
else:
    print('\nVector store not loaded -- run Cells 2-4 first')
    print('Keyword fallback available if corpus is loaded')

print('Vector store OK')

FAISS index loaded: 30 vectors


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7462.53it/s]



Test query: "glioblastoma treatment temozolomide"
Results: 3
  [0.570] NCCN Guidelines CNS Tumors v1.2024 -- Low-Grade Glioma Management. IDH-mutant grade 2 glioma: observation acceptable f...
  [0.372] NCCN Guidelines CNS Tumors v1.2024 -- GBM Standard of Care -- Stupp Protocol. Newly diagnosed GBM: maximal safe surgic...
  [0.304] NCCN Guidelines CNS Tumors v1.2024 -- Meningioma Management. Grade I meningioma: observation for asymptomatic, small (...
Vector store OK


---
## Cell 6 - RAG Query Pipeline

In [6]:
import os, json, urllib.request

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'


def rag_query(cancer_type, tumor_type, who_grade=None,
              patient_context='', top_k=4,
              api_key=None, use_claude=True):
    """
    Full RAG pipeline:
    1. Build retrieval query from clinical context
    2. Retrieve relevant chunks from vector store
    3. Format context for Claude
    4. Generate evidence-based recommendation

    Falls back to guidelines-only if Claude API unavailable.
    """
    # Step 1: Retrieve relevant context
    retrieval_query = (
        f'{tumor_type} {cancer_type} treatment '
        f'{who_grade or ""} NCCN guidelines therapy'
    )
    retrieved = vector_store.search(
        retrieval_query, top_k=top_k, cancer_type=cancer_type
    )

    if not retrieved:
        # Direct guideline lookup fallback
        retrieved = vector_store._keyword_search(
            tumor_type, top_k=top_k, cancer_type=cancer_type
        )

    # Step 2: Format retrieved context
    context_blocks = []
    for i, r in enumerate(retrieved, 1):
        block = (
            f'[Source {i}: {r["source"]}]\n'
            f'{r["text"]}'
        )
        context_blocks.append(block)
    context_str = '\n\n'.join(context_blocks)

    sources_cited = [
        {'source': r['source'], 'score': r.get('score', 0)}
        for r in retrieved
    ]

    # Step 3: Claude API synthesis
    api_key = api_key or os.environ.get('ANTHROPIC_API_KEY', '')
    if use_claude and api_key:
        prompt = (
            f'Clinical context:\n'
            f'  Cancer type : {cancer_type}\n'
            f'  Tumor type  : {tumor_type}\n'
            f'  WHO grade   : {who_grade or "unknown"}\n'
            f'  Patient     : {patient_context}\n'
            f'\nRetrieved evidence:\n{context_str}\n'
            f'\nBased on the retrieved evidence above, provide structured '
            f'treatment recommendations. Cite specific sources. '
            f'Format as JSON with keys: recommendations (list of strings), '
            f'reasoning (str), evidence_level (str), urgent_actions (list).'
        )

        try:
            payload = json.dumps({
                'model'     : 'claude-sonnet-4-20250514',
                'max_tokens': 1024,
                'system'    : (
                    'You are a clinical oncology AI assistant. '
                    'Provide evidence-based treatment recommendations using only '
                    'the retrieved context provided. Never diagnose. '
                    'Always recommend clinician review.'
                ),
                'messages'  : [{'role': 'user', 'content': prompt}],
            }).encode('utf-8')

            req = urllib.request.Request(
                'https://api.anthropic.com/v1/messages',
                data=payload,
                headers={
                    'Content-Type'     : 'application/json',
                    'x-api-key'        : api_key,
                    'anthropic-version': '2023-06-01',
                }
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                data    = json.loads(resp.read())
                content = data['content'][0]['text']
                try:
                    clean  = content.replace('```json', '').replace('```', '').strip()
                    result = json.loads(clean)
                except Exception:
                    result = {'recommendations': [content], 'reasoning': ''}
                result['source_mode']  = 'rag_claude'
                result['sources_cited'] = sources_cited
                return result

        except Exception as e:
            print(f'  Claude API failed: {e} -- falling back to retrieved context')

    # Step 4: Fallback -- return retrieved context directly
    recs = []
    for r in retrieved:
        text = r['text']
        # Extract sentences as recommendations
        for sent in text.split('.')[:3]:
            sent = sent.strip()
            if len(sent) > 20:
                recs.append(sent + '.')

    return {
        'recommendations': recs[:5],
        'reasoning'      : f'Based on retrieved: {[r["source"] for r in retrieved]}',
        'evidence_level' : retrieved[0].get('evidence', 'retrieved') if retrieved else 'none',
        'source_mode'    : 'rag_retrieval_only',
        'sources_cited'  : sources_cited,
    }


# ── Test RAG pipeline ─────────────────────────────────────────────────────────
print('Testing RAG pipeline...')
print()

test_cases = [
    ('brain',  'glioma',    'Grade IV (GBM)', 'age 55, male, MGMT methylated'),
    ('breast', 'malignant', None,             'age 48, female, HER2+'),
    ('skin',   'melanoma',  None,             'age 62, male, 2.1mm Breslow thickness'),
]

for cancer, tumor, grade, context in test_cases:
    print(f'{cancer.upper()} -- {tumor} {grade or ""}')
    result = rag_query(cancer, tumor, grade, context,
                       api_key=os.environ.get('ANTHROPIC_API_KEY'))
    print(f'  Mode    : {result["source_mode"]}')
    print(f'  Sources : {[s["source"][:40] for s in result.get("sources_cited", [])]}')
    print(f'  Recs    : {len(result.get("recommendations", []))} recommendations')
    if result.get('recommendations'):
        print(f'  First   : {result["recommendations"][0][:100]}')
    print()

Testing RAG pipeline...

BRAIN -- glioma Grade IV (GBM)
  Mode    : rag_retrieval_only
  Sources : ['NCCN Guidelines CNS Tumors v1.2024', 'NCCN Guidelines CNS Tumors v1.2024', 'NCCN Guidelines CNS Tumors v1.2024']
  Recs    : 5 recommendations
  First   : Low-Grade Glioma Management.

BREAST -- malignant 
  Mode    : rag_retrieval_only
  Sources : ['NCCN Guidelines Breast Cancer v4.2024', 'PubMed PMID:38640824', 'PubMed PMID:38787015', 'PubMed PMID:38977914']
  Recs    : 5 recommendations
  First   : Invasive Breast Cancer -- Surgical and Systemic Therapy.

SKIN -- melanoma 
  Mode    : rag_retrieval_only
  Sources : ['NCCN Guidelines Melanoma v2.2024', 'PubMed PMID:38905895', 'PubMed PMID:38796546', 'PubMed PMID:38742379']
  Recs    : 5 recommendations
  First   : Cutaneous Melanoma Treatment.



---
## Cell 7 - Upgraded Agent 5 with RAG

In [7]:
import os

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'


class ClinicalIntelligenceAgentRAG:
    """
    Agent 5 -- Upgraded with RAG.
    Three-tier fallback:
      1. RAG + Claude API (best -- retrieves evidence, reasons over it)
      2. Claude API alone (good -- no retrieved context)
      3. NCCN static rules (offline -- no API or internet)
    """

    def __init__(self, vector_store=None, api_key=None):
        self.vector_store = vector_store
        self.api_key      = api_key or os.environ.get('ANTHROPIC_API_KEY', '')
        self.use_rag      = vector_store is not None and bool(vector_store.corpus)
        self.use_claude   = bool(self.api_key)
        mode = ('RAG+Claude' if self.use_rag and self.use_claude
                else 'Claude-only' if self.use_claude
                else 'Rule-based')
        print(f'ClinicalIntelligenceAgentRAG initialized -- mode: {mode}')

    def recommend(self, cancer_type, tumor_type, who_grade=None,
                  patient_age=None, patient_sex=None,
                  clinical_notes='', confidence=0.0):
        """
        Generate treatment recommendation with full RAG pipeline.
        """
        patient_ctx = ' '.join(filter(None, [
            f'age {patient_age}' if patient_age else '',
            patient_sex or '',
            clinical_notes[:200] if clinical_notes else '',
        ]))

        if self.use_rag:
            return rag_query(
                cancer_type, tumor_type, who_grade,
                patient_ctx, top_k=4,
                api_key=self.api_key
            )

        # Fallback to static rules
        from nb11_rules import NCCN_RULES   # imported if available
        rules = NCCN_RULES.get(tumor_type) or NCCN_RULES.get('default')
        recs  = rules.get(who_grade) or rules.get('default') or list(rules.values())[0]
        return {
            'recommendations': recs,
            'reasoning'      : f'Static NCCN rules for {tumor_type}',
            'source_mode'    : 'static_rules',
            'sources_cited'  : [],
        }


# Initialize upgraded Agent 5
agent5_rag = ClinicalIntelligenceAgentRAG(
    vector_store=vector_store if loaded else None,
    api_key=os.environ.get('ANTHROPIC_API_KEY')
)

# Quick test
print()
result = agent5_rag.recommend(
    cancer_type='brain', tumor_type='glioma',
    who_grade='Grade IV (GBM)',
    patient_age=55, patient_sex='M',
    clinical_notes='MGMT methylated, IDH wildtype',
)
print(f'Mode     : {result["source_mode"]}')
print(f'Sources  : {len(result.get("sources_cited", []))}')
print(f'Recs     : {len(result.get("recommendations", []))}')
print('Agent 5 RAG OK')

ClinicalIntelligenceAgentRAG initialized -- mode: Rule-based

Mode     : rag_retrieval_only
Sources  : 3
Recs     : 5
Agent 5 RAG OK


---
## Cell 8 - Summary

In [8]:
import os

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'

print('=' * 65)
print('  NOTEBOOK 13 - RAG PIPELINE')
print('=' * 65)
print()
print('  Components built:')
print('    pubmed_search()              -- NCBI E-utilities with caching')
print('    NCCN/ESMO guidelines JSON    -- 8 structured guidelines')
print('    chunk_text()                 -- Overlapping text chunking')
print('    embed_corpus()               -- sentence-transformers embedding')
print('    MedicalVectorStore           -- FAISS + keyword fallback')
print('    rag_query()                  -- Retrieve + Claude synthesis')
print('    ClinicalIntelligenceAgentRAG -- Upgraded Agent 5')
print()
print('  Three-tier Agent 5 fallback:')
print('    1. RAG + Claude API   -- retrieves evidence, reasons over it')
print('    2. Claude API alone   -- no retrieved context')
print('    3. Static NCCN rules  -- fully offline')
print()

files = [
    os.path.join(BASE, 'rag', 'corpus',     'pubmed_corpus.json'),
    os.path.join(BASE, 'rag', 'guidelines', 'nccn_esmo_guidelines.json'),
    os.path.join(BASE, 'rag', 'index',      'embeddings.npy'),
    os.path.join(BASE, 'rag', 'index',      'corpus_meta.json'),
]
print('  Files created:')
for f in files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f)/(1024**2) if exists else 0
    print(f'    {os.path.basename(f):35s}: {"OK (" + f"{size:.1f}MB)" if exists else "not created"}')
print()
print('  Next: 14_Explainability_Engine.ipynb')
print('    - Grad-CAM for all CNNs')
print('    - MC Dropout uncertainty quantification')
print('    - Calibration curves')
print('    - Uncertainty heatmaps')
print('=' * 65)

  NOTEBOOK 13 - RAG PIPELINE

  Components built:
    pubmed_search()              -- NCBI E-utilities with caching
    NCCN/ESMO guidelines JSON    -- 8 structured guidelines
    chunk_text()                 -- Overlapping text chunking
    embed_corpus()               -- sentence-transformers embedding
    MedicalVectorStore           -- FAISS + keyword fallback
    rag_query()                  -- Retrieve + Claude synthesis
    ClinicalIntelligenceAgentRAG -- Upgraded Agent 5

  Three-tier Agent 5 fallback:
    1. RAG + Claude API   -- retrieves evidence, reasons over it
    2. Claude API alone   -- no retrieved context
    3. Static NCCN rules  -- fully offline

  Files created:
    pubmed_corpus.json                 : OK (0.0MB)
    nccn_esmo_guidelines.json          : OK (0.0MB)
    embeddings.npy                     : OK (0.0MB)
    corpus_meta.json                   : OK (0.0MB)

  Next: 14_Explainability_Engine.ipynb
    - Grad-CAM for all CNNs
    - MC Dropout uncertainty qua